# Generation run (Colab, T4)
Runtime -> Change runtime type -> **T4 GPU**. Run cells top to bottom. If Colab disconnects, re-run setup cells 1-3 and then the same generate cell: finished questions are skipped.

In [ ]:
# 1. Get the code into Drive (outputs live next to it, so they survive disconnects)
from google.colab import drive; drive.mount('/content/drive')
import os
PROJ = '/content/drive/MyDrive/coms4705-hallucination-gen'
if not os.path.exists(PROJ):
    REPO_URL = ''   # <- GitHub URL once the repo is pushed; leave '' to use the zip instead
    if REPO_URL:
        !git clone {REPO_URL} {PROJ}
    else:  # upload coms4705-hallucination-gen.zip to the top of My Drive first
        !unzip -q /content/drive/MyDrive/coms4705-hallucination-gen.zip -d /content/drive/MyDrive/
%cd {PROJ}
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Install
!pip install -q -r requirements.txt

In [ ]:
# 3. Hugging Face login (accept the Mistral-7B-Instruct-v0.2 license on huggingface.co first)
from huggingface_hub import login; login()

## Step A: 50 TriviaQA questions end to end (for Jenny and Seongjun)
About 15 min, mostly the ~15 GB model download.

In [ ]:
!python scripts/generate.py --dataset triviaqa --split train --limit 50

In [ ]:
!python -m pytest -s

In [ ]:
# Download this file and send it over to be committed and pushed
from google.colab import files; files.download('outputs/generations/triviaqa_train.jsonl')

## Step B: full run (~4,500 train/val + PopQA + GSM8K)
Long on a T4. Re-run this cell after any disconnect. Optional, for faster sampling: `!pip install vllm` first (it may not fit in T4 memory; if it errors, set `generation.sampling.backend: hf` in config.yaml).

In [ ]:
!python scripts/generate.py --dataset all --split all